# 02 - Deep Agent com planejamento

No notebook anterior, o deep agent respondeu a uma pergunta direta. Aqui ele recebe uma tarefa com etapas e precisa **planejar antes de executar**.

Fluxo:

```
START → planeja (write_todos) → executa a tarefa → resposta final → END
```

O planejamento usa a ferramenta `write_todos`. Ela cria uma lista de passos e atualiza o status de cada um. Essa ferramenta não vem por padrão no `create_deep_agent`, então o notebook a adiciona com o `TodoListMiddleware`, do LangChain.

## 1. Configuração

A chave e o modelo vêm do `.env` na raiz do projeto, como nos outros notebooks.

O rastreamento usa o MLflow com um banco SQLite, `mlflow.db`, criado na pasta deste notebook. O `mlflow.langchain.autolog()` registra automaticamente cada execução do agente, com as chamadas ao modelo e às ferramentas, como um *trace*.

In [1]:
import os

from dotenv import find_dotenv, load_dotenv
from IPython.display import Markdown, display
from langchain.agents.middleware import TodoListMiddleware
from langchain_core.messages import AIMessage, HumanMessage
from langchain_openai import ChatOpenAI

import mlflow

from deepagents import create_deep_agent

load_dotenv(find_dotenv())

mlflow.set_tracking_uri("sqlite:///mlflow.db")
EXPERIMENTO = "deep-agent-planejamento"
experimento = mlflow.set_experiment(EXPERIMENTO)
mlflow.langchain.autolog()

llm = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL", "gpt-5-mini"),
    api_key=os.getenv("OPENAI_API_KEY"),
)

2026/11/17 07:28:55 INFO mlflow.agent.hint: Load the `mlflow-trace-instrumentation` skill at /home/leonardo/Documentos/github/loop_engineering_study/.venv/lib/python3.13/site-packages/mlflow/assistant/skills/mlflow-trace-instrumentation/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


2026/11/17 07:28:55 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/11/17 07:28:55 INFO mlflow.store.db.utils: Updating database tables


2026/11/17 07:28:57 INFO mlflow.tracking.fluent: Experiment with name 'deep-agent-planejamento' does not exist. Creating a new experiment.


## 2. Criando o agente

O `system_prompt` exige a ordem: primeiro o plano com `write_todos`, depois a execução. O `TodoListMiddleware` adiciona a ferramenta `write_todos` ao agente.

In [2]:
agente = create_deep_agent(
    model=llm,
    middleware=[TodoListMiddleware()],
    system_prompt=(
        "Você é um assistente que resolve tarefas com etapas. "
        "Antes de responder, chame a ferramenta write_todos com um plano de 3 a 5 passos. "
        "Depois de planejar, execute os passos e escreva a resposta final em português."
    ),
)

## 3. Executando

A função `executar` roda a tarefa e mostra as mensagens na ordem em que aconteceram. A primeira chamada de `write_todos` mostra o plano. Chamadas seguintes atualizam o status dos passos.

In [3]:
def executar(tarefa: str) -> dict:
    estado = agente.invoke({"messages": [HumanMessage(content=tarefa)]})

    linhas = [f"**Tarefa:** {tarefa}", ""]
    plano_mostrado = False
    for mensagem in estado["messages"][1:]:
        if isinstance(mensagem, AIMessage) and mensagem.tool_calls:
            for chamada in mensagem.tool_calls:
                if chamada["name"] == "write_todos" and not plano_mostrado:
                    linhas.append("**Plano:**")
                    for item in chamada["args"].get("todos", []):
                        linhas.append(f"- {item.get('content', item)}")
                    linhas.append("")
                    plano_mostrado = True
                elif chamada["name"] == "write_todos":
                    linhas.append("→ `write_todos` (atualiza o status dos passos)")
                else:
                    linhas.append(f"→ `{chamada['name']}`")
        elif isinstance(mensagem, AIMessage) and mensagem.content:
            linhas.append("")
            linhas.append(f"**Resposta:** {mensagem.content}")
    display(Markdown("\n".join(linhas)))
    return estado


estado = executar(
    "Monte um roteiro de estudo de 3 dias para aprender LangGraph, com um tema por dia."
)

**Tarefa:** Monte um roteiro de estudo de 3 dias para aprender LangGraph, com um tema por dia.

**Plano:**
- Definir estrutura geral do roteiro e temas por dia
- Elaborar roteiro detalhado do Dia 1 (instalação e conceitos)
- Elaborar roteiro detalhado do Dia 2 (construção de grafos e integração com LLMs)
- Elaborar roteiro detalhado do Dia 3 (produção, deploy e avanços)

→ `write_todos` (atualiza o status dos passos)
→ `write_todos` (atualiza o status dos passos)
→ `write_todos` (atualiza o status dos passos)
→ `write_todos` (atualiza o status dos passos)

**Resposta:** Plano de estudos de 3 dias para aprender LangGraph

Objetivo: em 3 dias você terá uma compreensão prática dos conceitos centrais do LangGraph, saberá construir grafos de fluxo com LLMs, testar pipelines e preparar um protótipo pronto para produção.

Pré-requisitos
- Noções básicas de Python (funções, classes, pacotes).
- Conceitos básicos de LLMs (prompting, chamadas de API).
- Ambiente: Python 3.8+, Git, editor (VSCode/IDE), virtualenv/venv.
- Chaves de API para o provedor de LLM que você usará (ex.: OpenAI, Anthropic) e variável de ambiente configurada (por exemplo OPENAI_API_KEY).

Resumo dos temas por dia
- Dia 1 — Fundamentos e ambiente: conceitos de grafos, componentes do LangGraph, instalação e primeiro “hello world”.
- Dia 2 — Construção prática: criar grafos, conectar LLMs, manipular entradas/saídas e testes locais.
- Dia 3 — Produção e avanços: deploy, monitoramento, otimização e mini-projeto final.

Dia 1 — Fundamentos e ambiente (4–5 horas)
Objetivos do dia
- Entender o que é LangGraph, por que usar grafos para orquestrar LLMs e quais são os principais blocos (nós, arestas, pipelines, memoria/estado, ferramentas).
- Preparar ambiente e executar um exemplo simples.

Sessão A — Conceitos (60–90 min)
- Leia/assimile: conceitos de grafo aplicado a workflows com LLMs.
  - Nós = unidades de trabalho (ex.: prompt a um LLM, chamada a API, transformação).
  - Arestas = fluxo de dados entre nós.
  - Componentes recorrentes: agentes, ferramentas (retrievers, bases de dados), memória/state, condicionais/branching, loops.
- Resultado esperado: ser capaz de desenhar um fluxo simples no papel (ex.: input usuário -> LLM -> pós-processamento -> resposta).

Sessão B — Preparar ambiente (30–45 min)
- Criar ambiente virtual: python -m venv .venv && source .venv/bin/activate (ou equivalente Windows).
- Instalar LangGraph (verificar nome do pacote oficial); instalar também dependências comuns (ex.: requests, openai).
- Configurar variável de ambiente com sua chave de API.

Sessão C — Primeiro exemplo prático (90–120 min)
- Rodar um exemplo simples (ex.: “echo” via LLM) e inspecionar o grafo:
  - Carregar um grafo de exemplo se existir (ex.: exemplo de repositório/README).
  - Executar e observar logs/resultado.
- Exercício prático: criar um “Hello LangGraph” que receba texto e retorne uma versão resumida pelo LLM.
- Resultado: você terá um script que monta um grafo mínimo e o executa.

Materiais sugeridos
- Documentação oficial do LangGraph (guia de início rápido).
- Exemplos no repositório GitHub do projeto.

Dia 2 — Construção prática e integração com LLMs (5–6 horas)
Objetivos do dia
- Aprender a criar nós personalizados, conectar múltiplos nós em pipelines, integrar retrieval (busca em docs) e construir fluxos condicionais.

Sessão A — Nós, componentes e padrões (60–90 min)
- Tipos de nós:
  - Prompt nodes (enviam prompt ao LLM).
  - Transform nodes (processam/filtram texto).
  - IO nodes (leitura/escrita de DB ou arquivos).
  - Tool nodes (chamada a APIs externas: motores de busca, bases de dados).
- Padrões: Chain of Thought, Retrieval-Augmented Generation (RAG), decomposição de tarefas (task decomposition).

Sessão B — Hands-on: pipeline de QA com RAG (120–150 min)
- Objetivo prático: construir pipeline que:
  1. Recebe pergunta do usuário.
  2. Recupera documentos relevantes (retriever/FAISS / embeddings).
  3. Envia contexto + pergunta ao LLM para resposta final.
- Passos sugeridos:
  - Preparar um pequeno conjunto de documentos (ex.: 5-10 textos).
  - Indexar com ferramenta de vetores (ou mock se não houver lib).
  - Construir grafo: Input -> Retriever node -> Aggregator -> Prompt node -> Output.
- Testes: perguntas com e sem contexto relevante; medir qualidade e tempos.

Sessão C — Testes, debug e experimentação (60–90 min)
- Como instrumentar logs: ativar verbosidade, checar entradas/saídas de cada nó.
- Debugging: isolar nós, executar unit tests nos nós transform e retriever.
- Experimentos de prompting: testar variações de prompt, temperatura, truncamento de contexto.

Exercícios finais do dia
- Estender o pipeline para incluir:
  - Histórico de conversas (memória curta).
  - Branching: se o retriever não achar docs relevantes, acionar fallback diferente (ex.: pedir clarificação ao usuário).

Dia 3 — Produção, deploy e avanços (4–6 horas)
Objetivos do dia
- Preparar o grafo para produção, pensar em escalabilidade, observabilidade e segurança.
- Implementar um mini-projeto final integrando o que aprendeu.

Sessão A — Arquitetura para produção (60–90 min)
- Boas práticas:
  - Isolar nós em componentes testáveis e reusáveis.
  - Tratar erros e timeouts (circuit-breakers).
  - Limitar custos de LLM (batching, cache, controle de tokens).
  - Segurança: sanitização de input, políticas de rate limit, não logar chaves.
- Escalabilidade:
  - Separar compute-bound (LLM calls) dos IO-bound (DB).
  - Deploy em containers, serverless ou orquestradores (Kubernetes) dependendo da carga.

Sessão B — Observabilidade e monitoramento (45–60 min)
- Métricas importantes: latência por nó, taxa de erro, custo por chamada, cobertura de retriever (hit/miss).
- Ferramentas: logs estruturados, traces distribuídos (opentelemetry), dashboards.

Sessão C — Mini-projeto final (120–180 min)
- Sugestões de mini-projetos (escolha 1):
  1. Assistente de conhecimento para uma documentação interna (RAG + QA + histórico).
  2. Pipeline de resumo periódico: ingesta de novos documentos -> sumarização diária -> envio por email.
  3. Agente com ferramentas: pergunta do usuário -> se precisar acessar API externa, chamar ferramenta -> compor resposta.
- Entregáveis do mini-projeto:
  - Código do grafo com README.
  - Testes básicos e instruções de execução local.
  - Lista de melhorias futuras e estimativa de custos.
- Critérios de avaliação: funcionalidade, robustez (trata erros), observabilidade (logs/metricas) e documentação.

Materiais e próximos passos
- Revisar documentação oficial e exemplos do repositório GitHub.
- Experimentar com diferentes provedores de LLM e comparar custo/latência.
- Praticar criando bibliotecas de nós reutilizáveis e templates de grafos.
- Expandir para CI/CD e deploy (ex.: container + pipeline de testes + monitoramento).

Checklist rápido para cada dia (para acompanhar progresso)
- Dia 1: [ ] Entendi conceitos principais, [ ] Ambiente pronto, [ ] Rodei “hello” grafo.
- Dia 2: [ ] Construi pipeline de QA, [ ] Integrei retriever, [ ] Testei prompts e branches.
- Day 3: [ ] Pensei em produção, [ ] Configurei logs/metrics, [ ] Completei mini-projeto.

Se quiser, eu posso:
- Gerar um roteiro com horários mais detalhados (ex.: 9:00–10:30, 10:45–12:00...) adaptado à sua disponibilidade diária.
- Gerar exemplos de código passo a passo para cada exercício (instalação, pipeline RAG, mini-projeto).
- Sugerir bibliotecas específicas para embeddings/indexação (ex.: FAISS, Weaviate) e mostrar integração com LangGraph.

Qual opção prefere agora: roteiro com horários ou exemplos de código para os exercícios?

## 4. Acompanhando o rastreamento no MLflow

Cada execução acima virou um trace no experimento `deep-agent-planejamento`. Para ver:

1. Abra um terminal na pasta `notebooks/04-DeepAgents` (o caminho do `mlflow.db` é relativo a ela).
2. Rode:

```bash
uv run --project ../.. mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5000
```

3. Abra `http://127.0.0.1:5000` no navegador.
4. Em **Experiments**, escolha `deep-agent-planejamento` e abra a aba **Traces**.
5. Clique no trace mais recente. A árvore mostra os *spans*: cada chamada ao modelo, a chamada a `write_todos` e a resposta final. Nessa árvore, dá para conferir que o plano aparece antes da execução.
6. Pressione `Ctrl+C` no terminal para parar a interface.

A célula seguinte lista os traces e os spans pelo próprio notebook, sem abrir a interface.

## 5. Conferindo a ordem

Para confirmar que o planejamento veio antes da execução, listamos as chamadas de ferramenta em ordem. A primeira deve ser `write_todos`.

In [4]:
ordem = []
for mensagem in estado["messages"]:
    if isinstance(mensagem, AIMessage):
        ordem.extend(chamada["name"] for chamada in mensagem.tool_calls)

print(" → ".join(ordem) if ordem else "nenhuma ferramenta chamada")
print("Planejou primeiro:", bool(ordem) and ordem[0] == "write_todos")

write_todos → write_todos → write_todos → write_todos → write_todos
Planejou primeiro: True


## 6. Lendo os traces pelo notebook

`mlflow.search_traces` lista os traces do experimento. O último trace mostra os spans em ordem, com o nome e o tipo de cada um.

In [5]:
traces = mlflow.search_traces(experiment_ids=[experimento.experiment_id], max_results=10)
print(f"Traces registrados no experimento: {len(traces)}")

trace = mlflow.get_trace(traces.iloc[0]["trace_id"])
print(f"\nSpans do trace mais recente ({trace.info.trace_id}):")
for span in trace.data.spans:
    print(f"- {span.name} ({span.span_type})")

Traces registrados no experimento: 1

Spans do trace mais recente (tr-2f4b2a21fc6d3b6c508aa0c8004a32c1):
- LangGraph (CHAIN)
- PatchToolCallsMiddleware.before_agent (CHAIN)
- model (CHAIN)
- ChatOpenAI (CHAT_MODEL)
- TodoListMiddleware.after_model (CHAIN)
- tools (CHAIN)
- write_todos (TOOL)
- model (CHAIN)
- ChatOpenAI (CHAT_MODEL)
- TodoListMiddleware.after_model (CHAIN)
- tools (CHAIN)
- write_todos (TOOL)
- model (CHAIN)
- ChatOpenAI (CHAT_MODEL)
- TodoListMiddleware.after_model (CHAIN)
- tools (CHAIN)
- write_todos (TOOL)
- model (CHAIN)
- ChatOpenAI (CHAT_MODEL)
- TodoListMiddleware.after_model (CHAIN)
- tools (CHAIN)
- write_todos (TOOL)
- model (CHAIN)
- ChatOpenAI (CHAT_MODEL)
- TodoListMiddleware.after_model (CHAIN)
- tools (CHAIN)
- write_todos (TOOL)
- model (CHAIN)
- ChatOpenAI (CHAT_MODEL)
- TodoListMiddleware.after_model (CHAIN)
- _ParentNotifierMiddleware.after_agent (CHAIN)


/tmp/ipykernel_42179/313133131.py:1: FutureWarning: Parameter 'experiment_ids' is deprecated. Please use 'locations' instead.
  traces = mlflow.search_traces(experiment_ids=[experimento.experiment_id], max_results=10)


## Observações

- Sem o `TodoListMiddleware`, o agente não tem a ferramenta `write_todos`. O prompt sozinho não basta.
- O plano é só uma lista de passos. Quem garante a ordem é o `system_prompt`. Por isso a célula de conferência verifica a ordem das chamadas.
- O rastreamento é automático, pelo `mlflow.langchain.autolog()`. Não é preciso alterar o agente para gerar traces.
- Esta versão planeja e responde em sequência. Ferramentas e subagentes ficam para os próximos notebooks.